# SmartFlow — 02 · YOLOv8n training & evaluation

**What this notebook does:** for each dataset that passed notebook 01's data gate, it runs **one independent experiment**: fine-tune COCO-pretrained YOLOv8n, select the checkpoint on the **validation** split, then evaluate the selected checkpoint **once** on the held-out **test** split. Everything is saved to Google Drive: checkpoints, curves, confusion matrices, predictions, CSV/JSON metrics and configs.

**What it does not claim:** YOLOv8n alone does not reproduce the paper, which compares YOLOv7-tiny, YOLOv8-nano, YOLOv8-small and YOLO-NAS-small on its own private Karachi dataset. These are baseline experiments on public data, and their numbers are not comparable to the paper's 90% mAP / 74 FPS.

**The four planned experiments:**

| ID | Dataset | Runs? | Why |
|---|---|---|---|
| E1 | UA-DETRAC (HF mirror), 1 class | if the data gate passed | real boxes |
| E2 | BDD100K, 4 classes | if the data gate passed | real boxes |
| E3 | QMUL Junction | **skipped** | no bounding boxes exist; labels are never fabricated |
| E4 | AI City Challenge 2021 | **skipped** | no complete detection boxes (counts / ReID / anomaly / partial MTMC) |

**How to run:** *Runtime → Run all*. If Colab disconnects, run all again. Finished experiments are skipped, interrupted training resumes from `last.pt`, and nothing is overwritten.

**Outputs to inspect before continuing** (they are marked 🔍 in the cells):
1. **Cell 4:** GPU name and versions.
2. **Cell 10:** the experiment registry.
3. **Cell 11:** pre-training data checks.
4. **Cell 12:** the preflight checklist (all PASS).
5. **Cell 13:** the smoke test result and time estimate.
6. **Cells E1/E2:** training curves, best epoch, test metrics.
7. **Cell R:** the final summary.

## 1 · Configuration
Shared settings are applied **identically** to every experiment. They are fingerprinted, and the hash is part of the run name, so changing a setting starts a new run instead of resuming an incompatible one.

**Training subset (BDD100K).** BDD100K has 63,000 training images, about 9x UA-DETRAC, which is roughly 12–14 hours for 50 epochs on a free T4. So E2 trains on a **fixed random subset of 20,000 training images** (`train_subset`), drawn once with the seed and saved to `reports/`. Validation (7,000 images) and test (10,000 images) stay **complete**. The subset size is part of the run name and of every report, and must be stated in results. Set it to `None` to use all images.

Values taken from the paper (Table 7, YOLOv8-nano): AdamW optimizer, lr0 0.001, batch 16. The paper trained for 100 epochs on 2,182 images. 50 epochs is a documented Colab-budget choice for datasets of 7k–63k images; change `epochs` once here and it applies to all experiments. Settings the paper does not report, such as image size, pretrained weights and augmentation, use Ultralytics defaults and are recorded.

In [ ]:
# 1. Configuration
from pathlib import Path

CFG = {
    "drive_mount": "/content/drive",
    "drive_root": "/content/drive/MyDrive/SmartFlow",   # same as notebook 01
    "local_root": "/content/smartflow_work",
    "ultralytics_version": "8.4.160",
    "install_dependencies": True,
    "seed": 42,
    "allow_cpu_training": False,   # training requires a GPU; evaluation of finished runs also works on CPU
    # Experiments to STOP at their last saved epoch and evaluate now, e.g. ["E2"] when the free Colab GPU quota
    # runs out. best.pt (selected on validation among the completed epochs) is finalised and evaluated; the
    # report states the epoch it stopped at. A backup weights/last_before_finish.pt keeps the resumable state.
    "finish_early": ["E2"],        # E2 stopped here: free Colab GPU quota ran out (set [] to keep training)
    # Hide Ultralytics' per-batch progress bars: on large datasets they make the notebook too big for Colab
    # to save ("Timed out attempting to save"). One summary line per epoch is printed instead; warnings and
    # errors are still shown. Takes effect only before ultralytics is imported (restart the session if needed).
    "quiet_training": True,
    "train": {                     # identical for every experiment
        "model": "yolov8n.pt",     # COCO-pretrained YOLOv8n from Ultralytics
        "imgsz": 640,
        "epochs": 50,
        "batch": 16,
        "optimizer": "AdamW",
        "lr0": 0.001,
        "patience": 20,            # early stopping on VALIDATION mAP50-95 only
        "workers": 2,
        "cache": False,
        "amp": True,
        "deterministic": True,
        "close_mosaic": 10,
        "save_period": -1,
        "plots": True,
    },
    "eval": {                      # identical for every experiment
        "conf": 0.001,             # standard for mAP evaluation
        "iou": 0.7,                # NMS IoU (Ultralytics default)
        "batch": 16,
        "speed_images": 200,       # batch-1 timing on this many test images (after warm-up)
        "speed_warmup": 10,
        "pred_images": 12,         # test images saved with predicted boxes
        "pred_conf": 0.25,
    },
    "checks": {
        "full_verify_images": True,  # PIL verify() on every image (fast structural check)
        "decode_sample": 300,        # full decode of a random sample per split
    },
    "smoke": {                     # tiny pipeline test; never reported as a result
        "train_images": 64, "val_images": 16, "test_images": 16,
        "epochs": 2, "imgsz": 320, "batch": 8,
    },
    # Optional fixed training subset per dataset (Colab budget). The subset is drawn ONCE from the train split
    # with the seed and saved; val and test stay complete. None = use the full train split.
    "train_subset": {"ua_detrac_mirror": None, "bdd100k": 20000},
    "experiments": [               # four planned experiments, in order
        {"id": "E1", "dataset": "ua_detrac_mirror"},
        {"id": "E2", "dataset": "bdd100k"},
        {"id": "E3", "dataset": "qmul_junction"},
        {"id": "E4", "dataset": "aic2021"},
    ],
}
import os, sys
if CFG["quiet_training"]:
    if "ultralytics" in sys.modules and os.environ.get("YOLO_VERBOSE") != "False":
        print("NOTE: ultralytics is already imported; quiet mode applies after Runtime -> Restart session.")
    os.environ["YOLO_VERBOSE"] = "False"
print("Shared training settings:", CFG["train"])

## 2 · Pinned dependencies
This installs `ultralytics==8.4.160` only if it isn't already installed. It is the same pin as notebook 01, so the API used here (callbacks, `val(split=...)`, `summary()`) is the one that was tested. The cell warns if the install changed Colab's preinstalled torch or numpy.

In [ ]:
# 2. Pinned dependencies
import importlib.metadata as md, subprocess, sys
assert "CFG" in globals(), "Run cell 1 first."

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

WATCH = ["torch", "torchvision", "numpy", "opencv-python", "pillow", "pandas", "matplotlib", "ultralytics"]
before = {p: _ver(p) for p in WATCH}
if CFG["install_dependencies"] and before["ultralytics"] != CFG["ultralytics_version"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={CFG['ultralytics_version']}"],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert r.returncode == 0, "pip install failed - see above."
after = {p: _ver(p) for p in WATCH}
changed = {p: (before[p], after[p]) for p in WATCH if p != "ultralytics" and before[p] != after[p]}
for p in WATCH:
    print(f"{p:>14}: {after[p]}")
assert after["ultralytics"] == CFG["ultralytics_version"], \
    f"ultralytics {after['ultralytics']} installed, {CFG['ultralytics_version']} required"
if changed:
    print("\nWARNING: preinstalled packages changed:", changed, "-> Runtime > Restart session, then Run all.")

## 3 · Mount Google Drive

In [ ]:
# 3. Mount Google Drive
from pathlib import Path
assert "CFG" in globals(), "Run cell 1 first."
try:
    from google.colab import drive as _gdrive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    if not Path(CFG["drive_mount"], "MyDrive").is_dir():
        _gdrive.mount(CFG["drive_mount"])
    assert Path(CFG["drive_mount"], "MyDrive").is_dir(), "Drive mount failed - rerun this cell."
    print("Drive mounted.")
else:
    print("Not in Colab: using", CFG["drive_root"])

## 4 · Environment, directories, reproducibility 🔍
This records the GPU, CUDA, cuDNN, torch, ultralytics and Python versions, and fixes the seeds for Python, NumPy and torch. Ultralytics additionally receives `seed` and `deterministic=True`. The snapshot is saved to `logs/`.

**Inspect:** the GPU line. A T4 or better is expected; on CPU, full experiments are refused unless `allow_cpu_training` is set.

In [ ]:
# 4. Environment + directories + seeds
import json, os, platform, random, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import ultralytics
assert "CFG" in globals(), "Run cell 1 first."

DRIVE, LOCAL = Path(CFG["drive_root"]), Path(CFG["local_root"])
DIRS = {k: DRIVE / k for k in ("raw", "prepared_cache", "checkpoints", "logs", "metrics", "plots", "reports")}
DIRS.update(local_yolo=LOCAL / "yolo", local_tmp=LOCAL / "tmp", pretrained=DRIVE / "checkpoints" / "pretrained",
            smoke=DRIVE / "checkpoints" / "_smoke")
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
DEVICE = 0 if torch.cuda.is_available() else "cpu"
ENV = {
    "utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "torchvision": __import__("torchvision").__version__,
    "ultralytics": ultralytics.__version__, "numpy": np.__version__,
    "cuda_available": torch.cuda.is_available(), "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
                     if torch.cuda.is_available() else None,
    "driver": subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else None,
    "cpu_count": os.cpu_count(), "device": str(DEVICE), "seed": CFG["seed"],
}
for k, v in ENV.items():
    print(f"{k:>16}: {v}")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
(DIRS["logs"] / f"nb02_environment_{stamp}.json").write_text(json.dumps({"env": ENV, "cfg": CFG}, indent=2, default=str))
if not torch.cuda.is_available():
    print("\nWARNING: no GPU. Runtime -> Change runtime type -> GPU. Full experiments will refuse to start on CPU.")

## 5 · Generic helpers
Logging, PASS/WARN/FAIL reports, hashing with a cache, safe archive extraction, and JSON/CSV writers. These are standalone copies, so this notebook doesn't depend on notebook 01's runtime state.

In [ ]:
# 5. Generic helpers
import csv, hashlib, math, tarfile, time, traceback
from collections import Counter, defaultdict
assert "DIRS" in globals(), "Run cell 4 first."
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


class UserActionRequired(RuntimeError):
    """The notebook cannot continue without something only you can provide."""


def log(msg):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(DIRS["logs"] / "training.log", "a") as f:
        f.write(line + "\n")


def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]


def to_py(o):
    """Make numpy / torch / Path values JSON-serialisable; NaN -> None."""
    if isinstance(o, dict):
        return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, Path):
        return str(o)
    if hasattr(o, "item") and not isinstance(o, (str, bytes)):
        try:
            o = o.item()
        except (ValueError, AttributeError):
            return str(o)
    if isinstance(o, float) and not math.isfinite(o):
        return None
    return o


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(to_py(obj), indent=2))
    tmp.replace(path)


def write_csv(path, rows):
    rows = [to_py(r) for r in rows]
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


class Report:
    def __init__(self, name, folder=None):
        self.name, self.checks, self.info = name, [], {}
        self.folder = Path(folder or DIRS["reports"])

    def check(self, label, ok, detail="", fatal=True):
        ok = bool(ok)
        self.checks.append({"check": label, "ok": ok, "fatal": fatal, "detail": str(detail)[:2000]})
        print(f"  [{'PASS' if ok else ('FAIL' if fatal else 'WARN')}] {label}" + (f" -> {detail}" if detail != "" else ""))
        return ok

    def failures(self):
        return [c for c in self.checks if c["fatal"] and not c["ok"]]

    def finish(self, raise_on_fail=True):
        path = self.folder / f"{self.name}.json"
        write_json(path, {"name": self.name, "utc": datetime.now(timezone.utc).isoformat(),
                          "passed": not self.failures(), "info": self.info, "checks": self.checks})
        if self.failures() and raise_on_fail:
            raise AssertionError(f"{len(self.failures())} fatal check(s) failed in '{self.name}': "
                                 + "; ".join(c["check"] for c in self.failures()) + f". Report: {path}")
        print(f"  => {'all fatal checks passed' if not self.failures() else 'FAILED'} "
              f"({len(self.checks)} checks). Report: {path}")
        return not self.failures()


def file_hash(path, algo="sha256", chunk=16 << 20):
    path = Path(path)
    st = path.stat()
    side = path.with_name(path.name + f".{algo}.json")
    if side.exists():
        try:
            c = json.loads(side.read_text())
            if c["size"] == st.st_size and c["mtime_ns"] == st.st_mtime_ns:
                return c["hash"]
        except (ValueError, KeyError):
            pass
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    side.write_text(json.dumps({"size": st.st_size, "mtime_ns": st.st_mtime_ns, "hash": h.hexdigest()}))
    return h.hexdigest()


def safe_extract_tar(archive, dest):
    dest = Path(dest)
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    root = dest.resolve()
    with tarfile.open(archive) as tf:
        for m in tf.getmembers():
            t = (dest / m.name).resolve()
            if (t != root and root not in t.parents) or m.issym() or m.islnk():
                raise IOError(f"Unsafe member in {archive}: {m.name}")
        try:
            tf.extractall(dest, filter="data")
        except TypeError:
            tf.extractall(dest)


print("Helpers loaded.")

## 6 · Data preparation functions
* `load_gate()` reads `reports/data_gate.json` from notebook 01, and refuses to continue if the gate did not pass.
* `restore_dataset()` puts the validated YOLO dataset on local disk. It reuses a local copy only if its fingerprint matches; otherwise it extracts the Drive cache tar after verifying the tar's sha256.
* `verify_against_gate()` recomputes each split's file-list hash and compares it with the gate. Any image added, removed or moved between splits since validation is caught.
* `write_yamls()` writes **two** YAMLs. The **training YAML has no `test` key at all**, so the trainer can't see the test split. The **evaluation YAML** includes `test` and is used only after training.
* `pretrain_checks()` checks for empty splits, missing or orphan labels, class ids outside `names`, malformed or out-of-range boxes, and corrupt images. It also counts images and instances per class per split.

In [ ]:
# 6. Data preparation functions
import yaml
from PIL import Image


def load_gate():
    p = DIRS["reports"] / "data_gate.json"
    if not p.exists():
        raise UserActionRequired(f"{p} not found. Run notebook 01 first.")
    gate = json.loads(p.read_text())
    if not gate.get("passed"):
        raise UserActionRequired(f"Data gate did not pass (not ready: {gate.get('not_ready')}). Fix notebook 01 first.")
    return gate


def restore_dataset(variant, fingerprint):
    """Make the validated dataset available locally; returns its root."""
    root = DIRS["local_yolo"] / variant
    marker = root / ".prepared.json"
    local_note = ("no local copy" if not marker.exists() else
                  "local copy built with a different configuration")
    if marker.exists() and json.loads(marker.read_text()).get("fingerprint") == fingerprint:
        log(f"{variant}: local copy present and matches the gate fingerprint.")
        return root
    tar_path = DIRS["prepared_cache"] / f"{variant}.tar"
    man_path = DIRS["prepared_cache"] / f"{variant}.json"
    if not (tar_path.exists() and man_path.exists()):
        raise UserActionRequired(f"{variant}: {local_note}, and the Drive cache {tar_path.name} is missing from "
                                 f"{DIRS['prepared_cache']}. Usual cause: Google Drive was full when notebook 01 "
                                 "wrote it. Free Drive space, then rerun notebook 01 (it rebuilds and saves it).")
    man = json.loads(man_path.read_text())
    if man.get("fingerprint") != fingerprint:
        raise UserActionRequired(f"Drive cache for {variant} was built with a different configuration than the "
                                 "data gate records. Rerun notebook 01.")
    if tar_path.stat().st_size != man["size"] or file_hash(tar_path) != man["sha256"]:
        raise IOError(f"Drive cache {tar_path} is corrupt (size/sha256 mismatch). Rerun notebook 01.")
    log(f"{variant}: restoring from Drive cache ({man['size'] / 1024**3:.2f} GB) ...")
    tmp = DIRS["local_tmp"] / f"restore_{variant}"
    safe_extract_tar(tar_path, tmp)
    shutil.rmtree(root, ignore_errors=True)
    (tmp / variant).replace(root)
    shutil.rmtree(tmp, ignore_errors=True)
    assert json.loads(marker.read_text())["fingerprint"] == fingerprint
    return root


def split_files(root, split):
    d = Path(root) / "images" / split
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS) if d.is_dir() else []


def verify_against_gate(root, manifest, rep):
    for split, m in manifest.items():
        names = [p.name for p in split_files(root, split)]
        h = hashlib.sha256("\n".join(names).encode()).hexdigest()
        rep.check(f"{split}: file list identical to the validated gate manifest",
                  h == m["file_list_sha256"] and len(names) == m["images"], f"{len(names)} images")


def make_train_subset(root, n, tag):
    """Fixed seeded subset of the train split (images hard-linked, labels copied). Returns the subset root."""
    imgs = split_files(root, "train")
    assert 0 < n < len(imgs), f"train_subset={n} must be between 1 and {len(imgs) - 1}"
    pick = sorted(random.Random(CFG["seed"]).sample(imgs, n), key=lambda p: p.name)
    names_txt = "\n".join(p.name for p in pick)
    key = hashlib.sha256(names_txt.encode()).hexdigest()
    sub = DIRS["local_yolo"] / f"{Path(root).name}__train{n}_s{CFG['seed']}"
    marker = sub / ".subset.json"
    (DIRS["reports"] / f"{tag}_train_subset_{n}_s{CFG['seed']}.txt").write_text(names_txt + "\n")
    if marker.exists() and json.loads(marker.read_text()).get("file_list_sha256") == key:
        log(f"{tag}: training subset of {n} images already built ({sub}).")
        return sub, key
    shutil.rmtree(sub, ignore_errors=True)
    (sub / "images" / "train").mkdir(parents=True)
    (sub / "labels" / "train").mkdir(parents=True)
    for p in pick:
        try:
            os.link(p, sub / "images" / "train" / p.name)
        except OSError:
            shutil.copy2(p, sub / "images" / "train" / p.name)
        shutil.copy2(Path(root) / "labels" / "train" / f"{p.stem}.txt", sub / "labels" / "train" / f"{p.stem}.txt")
    write_json(marker, {"source": str(root), "n": n, "seed": CFG["seed"], "file_list_sha256": key})
    log(f"{tag}: built training subset of {n} / {len(imgs)} images -> {sub}")
    return sub, key


def write_yamls(root, names, tag, train_dir=None):
    """Training YAML (train+val only) and evaluation YAML (train+val+test)."""
    base = {"path": str(root), "train": str(train_dir) if train_dir else "images/train", "val": "images/val",
            "names": {i: n for i, n in enumerate(names)}}
    train_yaml = DIRS["local_yolo"] / f"{tag}__train.yaml"
    eval_yaml = DIRS["local_yolo"] / f"{tag}__eval.yaml"
    train_yaml.write_text("# training: NO test split on purpose\n" + yaml.safe_dump(base, sort_keys=False))
    eval_yaml.write_text("# evaluation only\n" + yaml.safe_dump({**base, "test": "images/test"}, sort_keys=False))
    assert "test" not in yaml.safe_load(train_yaml.read_text())
    return train_yaml, eval_yaml


def pretrain_checks(root, names, tag, check_cfg=None, splits=("train", "val", "test"), require_train_classes=True):
    """Return (report, counts). counts[split] = {images, empty_label_files, instances{class: n}}."""
    check_cfg = check_cfg or CFG["checks"]
    rep = Report(f"{tag}_pretrain_checks")
    counts = {}
    for split in splits:
        imgs = split_files(root, split)
        lbl_dir = Path(root) / "labels" / split
        lbls = {p.stem for p in lbl_dir.glob("*.txt")} if lbl_dir.is_dir() else set()
        stems = {p.stem for p in imgs}
        missing, orphan = sorted(stems - lbls), sorted(lbls - stems)
        inst, bad_cls, bad_box, empty = Counter(), [], [], 0
        for s in sorted(stems & lbls):
            lines = [l for l in (lbl_dir / f"{s}.txt").read_text().splitlines() if l.strip()]
            empty += not lines
            for ln in lines:
                parts = ln.split()
                try:
                    c = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except (ValueError, IndexError):
                    bad_box.append(f"{s}: {ln}")
                    continue
                if not 0 <= c < len(names):
                    bad_cls.append(f"{s}: class {c}")
                    continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                        and x - w / 2 >= -1e-6 and x + w / 2 <= 1 + 1e-6 and y - h / 2 >= -1e-6 and y + h / 2 <= 1 + 1e-6):
                    bad_box.append(f"{s}: {ln}")
                    continue
                inst[names[c]] += 1
        corrupt = []
        if check_cfg["full_verify_images"]:
            for p in imgs:
                try:
                    with Image.open(p) as im:
                        im.verify()
                except Exception as e:
                    corrupt.append(f"{p.name}: {e}")
        for p in random.Random(CFG["seed"]).sample(imgs, min(check_cfg["decode_sample"], len(imgs))):
            try:
                with Image.open(p) as im:
                    im.load()
            except Exception as e:
                corrupt.append(f"{p.name}: {e}")
        counts[split] = {"images": len(imgs), "empty_label_files": empty,
                         "instances": {n: inst.get(n, 0) for n in names}, "instances_total": sum(inst.values())}
        rep.check(f"{split}: split not empty", len(imgs) > 0, len(imgs))
        rep.check(f"{split}: no missing label files", not missing, missing[:5])
        rep.check(f"{split}: no orphan label files", not orphan, orphan[:5])
        rep.check(f"{split}: all class ids within names (0..{len(names) - 1})", not bad_cls, bad_cls[:5])
        rep.check(f"{split}: no malformed / out-of-range boxes", not bad_box, bad_box[:5])
        rep.check(f"{split}: no corrupt images "
                  + ("(verify all + decode sample)" if check_cfg["full_verify_images"] else "(decode sample)"),
                  not corrupt, corrupt[:5])
        for n in names:
            rep.check(f"{split}: class '{n}' has instances", inst.get(n, 0) > 0, inst.get(n, 0),
                      fatal=(split == "train" and require_train_classes))
    rep.info["counts"] = counts
    return rep, counts


print("Data preparation functions loaded.")

## 7 · Training functions
`train_or_resume()` is the only place that calls `model.train()`. Its decision logic is safe across disconnects. Every case is decided from the files on Drive, never from memory:

| State found on Drive | Action |
|---|---|
| `run_status.json` says `TRAINED`, and `best.pt` has been finalised | skip training |
| `last.pt` exists **with** optimizer state (interrupted run) | `YOLO(last.pt).train(resume=True)` |
| `last.pt` exists **without** optimizer state (training finished, but the status wasn't written) | mark `TRAINED`; **never** call resume. Tested: in 8.4.160, resuming a finished checkpoint silently starts a new training on the default `coco8` dataset |
| nothing | fresh training from the pretrained weights |

* **Per-epoch logging.** A callback logs each epoch to `epoch_log.jsonl` on Drive: validation fitness, whether `best.pt` was written, and epoch duration. It uses `on_model_save`, which fires only inside the training loop, unlike `on_fit_epoch_end`, which also fires during the final evaluation.
* **Checkpoint selection.** In 8.4.160 the fitness used to select `best.pt` is **validation mAP@50–95**. That was verified in the source: the weights are `[0, 0, 0, 1]`.
* **Out of memory.** Ultralytics halves the batch automatically on out-of-memory during the first epoch. The effective batch is read back from the checkpoint and reported. If it differs from the configured batch, the run is flagged as deviating from the shared settings.
* **Failures.** Any exception sets `FAILED` with the traceback and re-raises. A failed run is never reported as successful.

In [ ]:
# 7. Training functions
import pandas as pd
from ultralytics import YOLO

TRAIN_FP = stable_hash({"train": CFG["train"], "seed": CFG["seed"], "ultralytics": CFG["ultralytics_version"]})


def run_name(exp_id, variant, smoke=False):
    t = CFG["smoke"] if smoke else CFG["train"]
    m = Path(CFG["train"]["model"]).stem
    return f"{'SMOKE_' if smoke else ''}{exp_id}_{variant}__{m}_img{t['imgsz']}_e{t['epochs']}_s{CFG['seed']}_{TRAIN_FP}"


def read_status(run_dir):
    p = Path(run_dir) / "run_status.json"
    return json.loads(p.read_text()) if p.exists() else {"state": "NEW", "sessions": []}


def write_status(run_dir, **upd):
    st = read_status(run_dir)
    st.update(upd)
    st["updated_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(Path(run_dir) / "run_status.json", st)
    return st


def ckpt_state(path):
    """('missing'|'resumable'|'finished', checkpoint dict or None)."""
    path = Path(path)
    if not path.exists():
        return "missing", None
    ck = torch.load(path, map_location="cpu", weights_only=False)
    return ("finished" if ck.get("optimizer") is None else "resumable"), ck


def make_epoch_logger(run_dir):
    log_path = Path(run_dir) / "epoch_log.jsonl"
    pending = {}

    def on_model_save(tr):
        pending.clear()
        pending.update({"epoch": tr.epoch + 1, "fitness_val_map50_95": float(tr.fitness),
                        "best_fitness": float(tr.best_fitness), "best_saved": bool(tr.best_fitness == tr.fitness),
                        "batch": int(tr.batch_size), "utc": datetime.now(timezone.utc).isoformat()})

    def on_fit_epoch_end(tr):
        if pending.get("epoch") == tr.epoch + 1:          # skips the extra call made by final_eval()
            pending["epoch_time_s"] = float(tr.epoch_time) if tr.epoch_time else None
            with open(log_path, "a") as f:
                f.write(json.dumps(pending) + "\n")
            losses = tr.label_loss_items(tr.tloss) if tr.tloss is not None else {}
            print(f"  epoch {pending['epoch']}/{tr.epochs}  "
                  + "  ".join(f"{k.split('/')[1]}={v:.3f}" for k, v in losses.items())
                  + f"  val_mAP50={tr.metrics.get('metrics/mAP50(B)', float('nan')):.4f}"
                  + f"  val_mAP50-95={pending['fitness_val_map50_95']:.4f}"
                  + ("  *best*" if pending["best_saved"] else "")
                  + (f"  {pending['epoch_time_s']:.0f}s" if pending["epoch_time_s"] else ""), flush=True)
            pending.clear()

    return on_model_save, on_fit_epoch_end


def train_or_resume(exp_id, variant, train_yaml, project, smoke=False):
    """Train (or resume) one experiment. Returns the run directory. Raises on failure."""
    t = CFG["smoke"] if smoke else CFG["train"]
    name = run_name(exp_id, variant, smoke)
    run_dir = Path(project) / name
    run_dir.mkdir(parents=True, exist_ok=True)
    last, best = run_dir / "weights" / "last.pt", run_dir / "weights" / "best.pt"
    st = read_status(run_dir)
    state, _ = ckpt_state(last)
    if st["state"] in ("TRAINED", "EVALUATED") and ckpt_state(best)[0] == "finished":
        log(f"{name}: already trained -> skipping training.")
        return run_dir
    if state == "finished":
        log(f"{name}: last.pt is finalised (training completed before the status was written) -> not resuming.")
        write_status(run_dir, state="TRAINED")
        return run_dir
    if not smoke and exp_id in CFG["finish_early"]:
        if state != "resumable" or ckpt_state(best)[0] == "missing":
            raise UserActionRequired(f"{exp_id}: finish_early requested, but no interrupted training with a saved "
                                     "best.pt was found. Remove it from CFG['finish_early'].")
        res = pd.read_csv(run_dir / "results.csv")
        done = int(res["epoch"].max())
        shutil.copy2(last, run_dir / "weights" / "last_before_finish.pt")   # keeps the option to resume later
        from ultralytics.utils.torch_utils import strip_optimizer
        strip_optimizer(best)
        strip_optimizer(last)
        b_ck = ckpt_state(best)[1]
        write_status(run_dir, state="TRAINED", stopped_by_user_at_epoch=done,
                     effective_batch=int(b_ck["train_args"]["batch"]))
        log(f"{name}: training STOPPED BY USER after epoch {done}/{b_ck['train_args']['epochs']} "
            "(finish_early); best.pt finalised for evaluation.")
        return run_dir
    if not smoke and DEVICE == "cpu" and not CFG["allow_cpu_training"]:
        raise UserActionRequired("No GPU available, so training cannot continue. Runtime -> Change runtime type -> "
                                 "T4 GPU, then Run all (training resumes). If Colab says the GPU quota is used up: "
                                 "wait and retry later, or set CFG['finish_early'] = ['" + exp_id + "'] to stop "
                                 "here and evaluate the best checkpoint so far (works on CPU).")

    args = dict(data=str(train_yaml), epochs=t["epochs"], imgsz=t["imgsz"], batch=t["batch"],
                optimizer=CFG["train"]["optimizer"], lr0=CFG["train"]["lr0"], patience=CFG["train"]["patience"],
                workers=CFG["train"]["workers"], cache=CFG["train"]["cache"], amp=CFG["train"]["amp"],
                deterministic=CFG["train"]["deterministic"], close_mosaic=CFG["train"]["close_mosaic"],
                save_period=CFG["train"]["save_period"], plots=CFG["train"]["plots"], seed=CFG["seed"],
                device=DEVICE, project=str(project), name=name, exist_ok=True, val=True)
    sessions = st.get("sessions", [])
    session = {"start_utc": datetime.now(timezone.utc).isoformat(), "mode": "resume" if state == "resumable" else "fresh",
               "gpu": ENV["gpu"], "device": str(DEVICE)}
    write_status(run_dir, state="TRAINING", experiment=exp_id, variant=variant, smoke=smoke,
                 requested_args={**args, "model": CFG["train"]["model"]}, sessions=sessions + [session])
    on_save, on_end = make_epoch_logger(run_dir)
    t0 = time.time()
    try:
        if state == "resumable":
            log(f"{name}: resuming from {last}")
            model = YOLO(str(last))
            model.add_callback("on_model_save", on_save)
            model.add_callback("on_fit_epoch_end", on_end)
            model.train(resume=True, workers=CFG["train"]["workers"], device=DEVICE)
        else:
            log(f"{name}: fresh training from {CFG['train']['model']}")
            model = YOLO(str(PRETRAINED))
            model.add_callback("on_model_save", on_save)
            model.add_callback("on_fit_epoch_end", on_end)
            model.train(**args)
    except BaseException as e:
        session.update(end_utc=datetime.now(timezone.utc).isoformat(), wall_s=time.time() - t0,
                       error=f"{type(e).__name__}: {e}")
        write_status(run_dir, state="FAILED", sessions=sessions + [session],
                     error=traceback.format_exc()[-4000:], resumable=ckpt_state(last)[0] == "resumable")
        log(f"{name}: FAILED ({type(e).__name__}: {e}). Rerun this cell to resume from last.pt if it exists.")
        raise
    session.update(end_utc=datetime.now(timezone.utc).isoformat(), wall_s=time.time() - t0)
    b_state, b_ck = ckpt_state(best)
    if b_state != "finished":
        write_status(run_dir, state="FAILED", sessions=sessions + [session], error="best.pt missing or not finalised")
        raise RuntimeError(f"{name}: training returned but best.pt is {b_state}.")
    write_status(run_dir, state="TRAINED", sessions=sessions + [session],
                 effective_batch=int(b_ck["train_args"]["batch"]))
    log(f"{name}: training finished in {session['wall_s'] / 60:.1f} min (this session).")
    return run_dir


def training_record(run_dir):
    """Per-epoch table + summary, merged from results.csv (Ultralytics) and epoch_log.jsonl (ours)."""
    import pandas as pd
    run_dir = Path(run_dir)
    res = pd.read_csv(run_dir / "results.csv")
    res.columns = [c.strip() for c in res.columns]
    res = res.drop_duplicates("epoch", keep="last").sort_values("epoch")
    elog = {}
    p = run_dir / "epoch_log.jsonl"
    if p.exists():
        for line in p.read_text().splitlines():
            if line.strip():
                d = json.loads(line)
                elog[d["epoch"]] = d                      # keep the last entry per epoch (resume re-runs)
    res["epoch_time_s"] = res["epoch"].map(lambda e: elog.get(int(e), {}).get("epoch_time_s"))
    res["best_saved"] = res["epoch"].map(lambda e: elog.get(int(e), {}).get("best_saved"))
    m = "metrics/mAP50-95(B)"
    best_by_csv = int(res.loc[res[m] == res[m].max(), "epoch"].max())    # ties: best.pt keeps the later epoch
    # The callback log can miss an epoch if Colab disconnected between saving it and logging it; only trust the
    # log when it covers every epoch in results.csv, otherwise use results.csv (same rule as best.pt: last max).
    log_complete = set(int(e) for e in res["epoch"]) <= set(elog)
    saved = [e for e, d in elog.items() if d.get("best_saved")]
    best_by_log = max(saved) if (saved and log_complete) else None
    _, b_ck = ckpt_state(run_dir / "weights" / "best.pt")
    st = read_status(run_dir)
    total_wall = sum(s.get("wall_s", 0) for s in st.get("sessions", []))
    summary = {
        "epochs_completed": int(res["epoch"].max()), "epochs_planned": int(b_ck["train_args"]["epochs"]),
        "early_stopped": int(res["epoch"].max()) < int(b_ck["train_args"]["epochs"]),
        "best_epoch": best_by_log if best_by_log is not None else best_by_csv,
        "best_epoch_by_results_csv": best_by_csv, "best_epoch_by_callback_log": best_by_log,
        "best_val_map50_95_during_training": float(res[m].max()),
        "selection_metric": "validation mAP50-95 (Ultralytics 8.4.160 fitness)",
        "epoch_time_mean_s": float(res["epoch_time_s"].dropna().mean()) if res["epoch_time_s"].notna().any() else None,
        "training_time_sum_epochs_s": float(res["epoch_time_s"].dropna().sum()) if res["epoch_time_s"].notna().any() else None,
        "training_wall_time_all_sessions_s": total_wall,
        "sessions": len(st.get("sessions", [])),
        "effective_batch": int(b_ck["train_args"]["batch"]),
        "requested_batch": int(st.get("requested_args", {}).get("batch", b_ck["train_args"]["batch"])),
        "best_checkpoint": str(run_dir / "weights" / "best.pt"), "last_checkpoint": str(run_dir / "weights" / "last.pt"),
        "epoch_times_missing": int(res["epoch_time_s"].isna().sum()),
        "stopped_by_user_at_epoch": st.get("stopped_by_user_at_epoch"),
        "stop_reason": ("stopped by user (compute budget, finish_early)" if st.get("stopped_by_user_at_epoch")
                        else "early stopping (validation patience)" if int(res["epoch"].max()) < int(b_ck["train_args"]["epochs"])
                        else "completed full epoch budget"),
    }
    return res, summary


print("Training functions loaded. Training-config fingerprint:", TRAIN_FP)

## 8 · Evaluation functions
The same procedure is used for every experiment. The **finalised `best.pt`** (selected on validation) is evaluated with the official `model.val()`, at `conf=0.001` and NMS IoU 0.7:
1. on the **val** split, as the validation result that was used for selection;
2. on the **test** split, **once**, as the final held-out result. Nothing is selected or tuned after this step.

Metric definitions:
* **mAP@50 and mAP@50–95:** COCO-style mean over classes.
* **Precision and recall:** Ultralytics' values at the confidence that maximises mean F1.
* **F1:** computed from those P and R as F1 = 2PR/(P+R).
* **Per-class rows:** come from `metrics.summary()`. A class with no instances in a split gets an explicit 0-instance row with empty metrics.
* **Instance counts:** Ultralytics' instance counts are cross-checked against our own label counts, so a silently dropped label is detected.

Efficiency metrics:
* **Speed:** batch-1 `predict()` on up to 200 test images after a warm-up, at full FP32 precision on the Colab GPU. Pre-processing, inference and post-processing times are reported. This is **not** a Jetson / TensorRT number and is not comparable to the paper's edge FPS.
* **Model size:** size of `best.pt` in MB, parameter count and GFLOPs.

In [ ]:
# 8. Evaluation functions
from ultralytics.utils.torch_utils import get_flops, get_num_params


def f1_score(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def evaluate_split(weights, eval_yaml, names, split, out_dir, label_counts):
    t = CFG["eval"]
    model = YOLO(str(weights))
    m = model.val(data=str(eval_yaml), split=split, imgsz=EVAL_IMGSZ, batch=t["batch"], conf=t["conf"], iou=t["iou"],
                  device=DEVICE, plots=True, project=str(out_dir), name=f"eval_{split}", exist_ok=True, verbose=False)
    P, R, map50, map5095 = (float(x) for x in m.mean_results())
    overall = {"split": split, "class": "all", "images": label_counts[split]["images"],
               "instances": label_counts[split]["instances_total"],
               "precision": P, "recall": R, "f1": f1_score(P, R), "mAP50": map50, "mAP50_95": map5095}
    per_class, seen = [], set()
    for row in m.summary():
        seen.add(row["Class"])
        per_class.append({"split": split, "class": row["Class"], "images": int(row["Images"]),
                          "instances": int(row["Instances"]), "precision": float(row["Box-P"]),
                          "recall": float(row["Box-R"]), "f1": float(row["Box-F1"]),
                          "mAP50": float(row["mAP50"]), "mAP50_95": float(row["mAP50-95"])})
    for n in names:
        if n not in seen:
            per_class.append({"split": split, "class": n, "images": 0, "instances": 0, "precision": None,
                              "recall": None, "f1": None, "mAP50": None, "mAP50_95": None,
                              "note": "no instances in this split"})
    ul_counts = {names[i]: int(c) for i, c in enumerate(m.nt_per_class)} if m.nt_per_class is not None else {}
    mismatch = {n: (label_counts[split]["instances"][n], ul_counts.get(n)) for n in names
                if label_counts[split]["instances"][n] != ul_counts.get(n, 0)}
    return {"overall": overall, "per_class": per_class, "speed_ms_val": dict(m.speed),
            "save_dir": str(m.save_dir), "instance_count_mismatch": mismatch}


def benchmark_speed(weights, image_paths):
    t = CFG["eval"]
    model = YOLO(str(weights))
    paths = [str(p) for p in image_paths[: t["speed_images"] + t["speed_warmup"]]]
    rows = []
    for i, r in enumerate(model.predict(source=paths, imgsz=EVAL_IMGSZ, batch=1, device=DEVICE,   # FP32 (default)
                                        conf=t["pred_conf"], verbose=False, stream=True)):
        if i >= t["speed_warmup"]:
            rows.append(r.speed)
    if not rows:
        return {"images_timed": 0}
    mean = {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}
    total = sum(mean.values())
    return {"images_timed": len(rows), "batch": 1, "precision": "fp32", "device": ENV["gpu"] or "cpu",
            "imgsz": EVAL_IMGSZ, **{f"{k}_ms": v for k, v in mean.items()}, "total_ms": total,
            "fps_end_to_end": 1000 / total if total else None,
            "fps_inference_only": 1000 / mean["inference"] if mean.get("inference") else None}


def model_stats(weights):
    model = YOLO(str(weights))
    return {"weights_file_mb": Path(weights).stat().st_size / 1024**2, "parameters": int(get_num_params(model.model)),
            "gflops_at_imgsz": float(get_flops(model.model, EVAL_IMGSZ)), "imgsz": EVAL_IMGSZ}


def save_prediction_grid(weights, image_paths, out_png, out_dir):
    import matplotlib.pyplot as plt
    t = CFG["eval"]
    pick = random.Random(CFG["seed"]).sample(list(image_paths), min(t["pred_images"], len(image_paths)))
    model = YOLO(str(weights))
    res = model.predict(source=[str(p) for p in pick], imgsz=EVAL_IMGSZ, conf=t["pred_conf"], device=DEVICE,
                        save=True, project=str(out_dir), name="test_predictions", exist_ok=True, verbose=False)
    cols = 4
    rows = math.ceil(len(res) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.2 * rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, r in zip(np.atleast_1d(axes).ravel(), res):
        ax.imshow(r.plot()[..., ::-1])
        ax.set_title(f"{Path(r.path).name[:40]} ({len(r.boxes)} det)", fontsize=7)
    fig.suptitle(f"Test-set predictions (conf >= {t['pred_conf']}) - qualitative only", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_png, dpi=100)
    plt.show()
    plt.close(fig)


print("Evaluation functions loaded.")

## 9 · Reporting functions
For each experiment, the reporting step writes the following into `<run_dir>/report/`:
* `metrics.json`: everything about the run.
* `per_class_metrics.csv`, `epochs.csv` and `training_config.json`.
* Loss and mAP curves, prediction grid, confusion matrices (raw and normalised) and the PR curve. Key figures are also copied to `plots/`.

After every experiment it rebuilds the cross-experiment tables `metrics/summary_all_experiments.{csv,json}` and `metrics/per_class_all_experiments.csv`. SKIPPED and FAILED experiments appear there with their reason and **no metric values**.

In [ ]:
# 9. Reporting functions
import pandas as pd


def plot_curves(epochs_df, out_png, title):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
    for ax, comp in zip(axes[:3], ("box", "cls", "dfl")):
        ax.plot(epochs_df["epoch"], epochs_df[f"train/{comp}_loss"], label="train")
        ax.plot(epochs_df["epoch"], epochs_df[f"val/{comp}_loss"], label="val")
        ax.set_title(f"{comp} loss"); ax.set_xlabel("epoch"); ax.legend()
    ax = axes[3]
    ax.plot(epochs_df["epoch"], epochs_df["metrics/mAP50(B)"], label="val mAP50")
    ax.plot(epochs_df["epoch"], epochs_df["metrics/mAP50-95(B)"], label="val mAP50-95")
    ax.set_title("validation mAP"); ax.set_xlabel("epoch"); ax.legend()
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(out_png, dpi=100)
    plt.show()
    plt.close(fig)


def build_report(exp, run_dir, counts, train_summary, epochs_df, val_res, test_res, speed, stats, smoke=False):
    run_dir = Path(run_dir)
    rep_dir = run_dir / "report"
    rep_dir.mkdir(exist_ok=True)
    st = read_status(run_dir)
    record = {
        "experiment": exp["id"], "dataset": exp["dataset"], "variant": exp["variant"], "smoke_test": smoke,
        "status": "EVALUATED", "model": CFG["train"]["model"], "run_dir": str(run_dir),
        "classes": exp["names"], "label_version": exp.get("labels"),
        "data_counts": counts, "training": train_summary, "validation": val_res, "test": test_res,
        "speed": speed, "model_stats": stats, "environment": ENV,
        "training_config": {**(CFG["smoke"] if smoke else CFG["train"]), "requested_args": st.get("requested_args")},
        "eval_config": CFG["eval"], "data_gate_fingerprint": exp["fingerprint"],
        "train_subset": exp.get("subset"), "notes": exp.get("notes", []),
    }
    write_json(rep_dir / "metrics.json", record)
    write_json(rep_dir / "training_config.json", record["training_config"])
    write_csv(rep_dir / "per_class_metrics.csv", val_res["per_class"] + test_res["per_class"]
              + [val_res["overall"], test_res["overall"]])
    epochs_df.to_csv(rep_dir / "epochs.csv", index=False)
    if not smoke:
        tag = f"{exp['id']}_{exp['variant']}"
        for src in (rep_dir / "curves.png", rep_dir / "test_predictions.png",
                    Path(test_res["save_dir"]) / "confusion_matrix.png",
                    Path(test_res["save_dir"]) / "confusion_matrix_normalized.png",
                    Path(test_res["save_dir"]) / "BoxPR_curve.png"):
            if src.exists():
                shutil.copy2(src, DIRS["plots"] / f"{tag}__{src.parent.name}_{src.name}")
    write_status(run_dir, state="EVALUATED")
    return record


def summary_row(exp, record=None, status=None, reason=None):
    row = {"experiment": exp["id"], "dataset": exp["dataset"], "variant": exp.get("variant"),
           "status": status or (record or {}).get("status"), "reason": reason}
    if record:
        tr, te, va = record["training"], record["test"]["overall"], record["validation"]["overall"]
        row.update({
            "classes": "|".join(record["classes"]), "label_version": record.get("label_version"),
            "train_images": record["data_counts"]["train"]["images"],
            "train_images_full": record["data_counts"].get("train_full", record["data_counts"]["train"])["images"],
            "val_images": record["data_counts"]["val"]["images"],
            "test_images": record["data_counts"]["test"]["images"],
            "test_instances": record["data_counts"]["test"]["instances_total"],
            "val_mAP50_95": va["mAP50_95"],
            "test_precision": te["precision"], "test_recall": te["recall"], "test_f1": te["f1"],
            "test_mAP50": te["mAP50"], "test_mAP50_95": te["mAP50_95"],
            "best_epoch": tr["best_epoch"], "epochs_completed": tr["epochs_completed"],
            "early_stopped": tr["early_stopped"], "stop_reason": tr.get("stop_reason"), "training_time_h": (tr["training_time_sum_epochs_s"] or 0) / 3600,
            "effective_batch": tr["effective_batch"],
            "speed_total_ms": record["speed"].get("total_ms"), "fps_end_to_end": record["speed"].get("fps_end_to_end"),
            "speed_device": record["speed"].get("device"), "weights_mb": record["model_stats"]["weights_file_mb"],
            "parameters": record["model_stats"]["parameters"], "gflops": record["model_stats"]["gflops_at_imgsz"],
            "best_checkpoint": tr["best_checkpoint"],
        })
    return row


def rebuild_global_tables():
    rows, per_class = [], []
    for exp in EXPERIMENTS:
        if exp["status"] in ("SKIPPED", "DATA_UNAVAILABLE"):
            rows.append(summary_row(exp, status=exp["status"], reason=exp["reason"]))
            continue
        run_dir = DIRS["checkpoints"] / run_name(exp["id"], exp["run_variant"])
        mfile = run_dir / "report" / "metrics.json"
        st = read_status(run_dir)
        if st["state"] == "EVALUATED" and mfile.exists():
            rec = json.loads(mfile.read_text())
            rows.append(summary_row(exp, rec))
            for r in rec["validation"]["per_class"] + rec["test"]["per_class"]:
                per_class.append({"experiment": exp["id"], "variant": exp["variant"], **r})
        else:
            rows.append(summary_row(exp, status=st["state"], reason=(st.get("error") or "")[-300:] or "not run yet"))
    write_csv(DIRS["metrics"] / "summary_all_experiments.csv", rows)
    write_json(DIRS["metrics"] / "summary_all_experiments.json", rows)
    if per_class:
        write_csv(DIRS["metrics"] / "per_class_all_experiments.csv", per_class)
    return pd.DataFrame(rows)


print("Reporting functions loaded.")

## 10 · Experiment registry 🔍
Experiments are built from the **data gate**: only datasets with status `PASS` are runnable. QMUL Junction and AI City 2021 are recorded as `SKIPPED` with their reason, and no labels are created for them. Each runnable experiment gets a unique run name and directory, `checkpoints/<run_name>/`.

**Inspect:** E1 and E2 should be `RUN`, and E3 and E4 should be `SKIPPED`.

In [ ]:
# 10. Experiment registry
GATE = load_gate()
EXPERIMENTS = []
for e in CFG["experiments"]:
    ds = e["dataset"]
    if ds in GATE["datasets"] and GATE["datasets"][ds]["status"] == "PASS":
        g = GATE["datasets"][ds]
        ry = yaml.safe_load((DIRS["reports"] / f"{g['variant']}.yaml").read_text())
        names = [ry["names"][i] for i in sorted(ry["names"])]
        n_sub = CFG["train_subset"].get(ds)
        EXPERIMENTS.append({**e, "status": "RUN", "variant": g["variant"], "names": names,
                            "run_variant": g["variant"] + (f"_train{n_sub}" if n_sub else ""), "train_subset": n_sub,
                            "fingerprint": g["fingerprint"], "manifest": g["manifest"], "labels": g.get("labels")})
    else:
        reason = GATE.get("excluded", {}).get(ds) or f"not PASS in the data gate ({GATE['datasets'].get(ds, {}).get('status', 'absent')})"
        EXPERIMENTS.append({**e, "status": "SKIPPED", "reason": reason})
for e in EXPERIMENTS:
    if e["status"] == "RUN":
        print(f"{e['id']}  RUN      {e['variant']:<24} classes={e['names']}  labels={e.get('labels')}  "
              f"train={'subset of ' + str(e['train_subset']) + ' images' if e['train_subset'] else 'full'}")
        print(f"{'':>13}run dir: {DIRS['checkpoints'] / run_name(e['id'], e['run_variant'])}")
    else:
        print(f"{e['id']}  SKIPPED  {e['dataset']:<24} {e['reason']}")
assert any(e["status"] == "RUN" for e in EXPERIMENTS), "No runnable experiment - check notebook 01."
assert len({run_name(e["id"], e["run_variant"]) for e in EXPERIMENTS if e["status"] == "RUN"}) == \
       sum(e["status"] == "RUN" for e in EXPERIMENTS), "run names must be unique"

## 11 · Restore data and run pre-training checks 🔍
For every runnable experiment, the cell restores the dataset from the Drive cache if needed and verifies the file lists against the gate. It then writes the training and evaluation YAMLs and runs `pretrain_checks()`.

**Inspect:**
* every line must be `PASS`;
* the per-split image and instance table;
* any `WARN` for a class with 0 instances in val or test, because that class's metrics will be empty there.

In [ ]:
# 11. Restore + pre-training checks
PREP = {}
for e in EXPERIMENTS:
    if e["status"] != "RUN":
        continue
    print(f"\n=== {e['id']} {e['variant']}")
    try:
        root = restore_dataset(e["variant"], e["fingerprint"])
    except (UserActionRequired, IOError) as err:
        e.update(status="DATA_UNAVAILABLE", reason=str(err))
        print(f"  [FAIL] {e['id']} cannot run: {err}\n  -> {e['id']} is marked DATA_UNAVAILABLE; "
              "the other experiments continue.")
        continue
    rep, counts = pretrain_checks(root, e["names"], f"{e['id']}_{e['variant']}")
    verify_against_gate(root, e["manifest"], rep)
    rep.finish()
    train_dir, subset_info = None, None
    if e["train_subset"]:
        sub_root, key = make_train_subset(root, e["train_subset"], f"{e['id']}_{e['variant']}")
        srep, scounts = pretrain_checks(sub_root, e["names"], f"{e['id']}_{e['variant']}_train_subset", splits=("train",))
        srep.check("subset is a strict subset of the validated train split",
                   {p.name for p in split_files(sub_root, "train")} <= {p.name for p in split_files(root, "train")})
        srep.finish()
        counts = {**counts, "train_full": counts["train"], "train": scounts["train"]}
        train_dir = sub_root / "images" / "train"
        subset_info = {"n": e["train_subset"], "of": counts["train_full"]["images"], "seed": CFG["seed"],
                       "file_list_sha256": key,
                       "list_file": str(DIRS["reports"] / f"{e['id']}_{e['variant']}_train_subset_{e['train_subset']}_s{CFG['seed']}.txt")}
    train_yaml, eval_yaml = write_yamls(root, e["names"], f"{e['id']}_{e['run_variant']}", train_dir)
    PREP[e["id"]] = {"root": root, "train_yaml": train_yaml, "eval_yaml": eval_yaml, "counts": counts,
                     "subset": subset_info}
    table = pd.DataFrame({s: {"images": c["images"], "empty_label_files": c["empty_label_files"], **c["instances"],
                              "instances_total": c["instances_total"]} for s, c in counts.items()})
    print(table.to_string())
unavailable = [e["id"] for e in EXPERIMENTS if e["status"] == "DATA_UNAVAILABLE"]
assert PREP, "No dataset could be prepared - see the messages above."
if unavailable:
    print(f"\nWARNING: {unavailable} cannot run until their data is available; the others will run.")

## 12 · Preflight checklist 🔍
Every item must be **PASS** before the real experiments start:

| # | Check |
|---|---|
| 1 | GPU available (or CPU explicitly allowed) |
| 2 | `ultralytics` is exactly the pinned version |
| 3 | Data gate passed and at least one experiment is runnable |
| 4 | Every runnable dataset passed the pre-training checks (cell 11) |
| 5 | Training YAMLs contain **no** `test` key; evaluation YAMLs do |
| 6 | Pretrained weights are present: downloaded once to Drive, sha256 recorded |
| 7 | Drive is writable and has free space; local disk has free space |
| 8 | The shared training config is identical for all experiments (single fingerprint) |
| 9 | Seeds are set and `deterministic=True` |

In [ ]:
# 12. Preflight checklist
from ultralytics.utils.downloads import attempt_download_asset
rep = Report("nb02_preflight")
rep.check("1. GPU available (required to train; finished runs can be evaluated on CPU)",
          torch.cuda.is_available() or CFG["allow_cpu_training"], ENV["gpu"] or "CPU - training will refuse to start",
          fatal=False)
rep.check("2. ultralytics version pinned", ultralytics.__version__ == CFG["ultralytics_version"], ultralytics.__version__)
rep.check("3. data gate passed, >=1 runnable experiment", GATE["passed"] and any(e["status"] == "RUN" for e in EXPERIMENTS))
runnable = [e for e in EXPERIMENTS if e["status"] == "RUN"]
rep.check("data available for every gated experiment", not [e for e in EXPERIMENTS if e["status"] == "DATA_UNAVAILABLE"],
          [e["id"] for e in EXPERIMENTS if e["status"] == "DATA_UNAVAILABLE"], fatal=False)
rep.check("4. pre-training checks passed for every runnable dataset", all(e["id"] in PREP for e in runnable),
          [e["id"] for e in runnable if e["id"] not in PREP])
ok5 = all("test" not in yaml.safe_load(PREP[e["id"]]["train_yaml"].read_text())
          and "test" in yaml.safe_load(PREP[e["id"]]["eval_yaml"].read_text()) for e in runnable if e["id"] in PREP)
rep.check("5. training YAML has no test split; evaluation YAML has it", ok5)
PRETRAINED = DIRS["pretrained"] / CFG["train"]["model"]
if not PRETRAINED.exists():
    cwd = os.getcwd()
    os.chdir(DIRS["pretrained"])
    try:
        got = Path(attempt_download_asset(CFG["train"]["model"])).resolve()   # official Ultralytics release asset
    finally:
        os.chdir(cwd)
    if got != PRETRAINED.resolve() and got.exists():
        shutil.copy2(got, PRETRAINED)
pre_sha = file_hash(PRETRAINED) if PRETRAINED.exists() else None
rep.check("6. pretrained weights present", PRETRAINED.exists(), f"{PRETRAINED} sha256={pre_sha}")
free_drive = shutil.disk_usage(DIRS["checkpoints"]).free / 1024**3
free_local = shutil.disk_usage(DIRS["local_yolo"]).free / 1024**3
probe = DIRS["checkpoints"] / ".write_test"
probe.write_text("ok")
rep.check("7. Drive writable, >=2 GB free; local >=5 GB free", probe.read_text() == "ok" and free_drive > 2 and free_local > 5,
          f"drive {free_drive:.1f} GB, local {free_local:.1f} GB")
probe.unlink()
rep.check("8. one shared training config for all experiments", len({TRAIN_FP}) == 1, f"fingerprint {TRAIN_FP}: {CFG['train']}")
rep.check("9. seed set and deterministic=True", CFG["train"]["deterministic"] and CFG["seed"] is not None, f"seed={CFG['seed']}")
rep.info.update(pretrained_sha256=pre_sha, env=ENV)
rep.finish()

## 13 · Smoke test (tiny subset) 🔍
This runs the **whole pipeline end to end** on a tiny random subset of each runnable dataset: by default 64 train, 16 val and 16 test images, 2 epochs at 320 px, batch 8. It covers checks, training, val and test evaluation, speed, the reporting step and file outputs. The subsets are drawn from each split separately, so no leakage is possible.

Outputs go to `checkpoints/_smoke/` and are marked `smoke_test: true`. **They are never results.** Accuracy after 2 epochs on 64 images is meaningless and is not judged. The only checks are that the pipeline ran and produced every expected file.

The smoke test is skipped on rerun once it has passed for the current configuration. It also measures training throughput and prints a **rough time estimate** for the full experiments.

**Inspect:** `SMOKE TEST PASSED` and the time estimate. If the estimate is too long for your Colab budget, lower `CFG["train"]["epochs"]`, which applies to all experiments equally, before continuing.

In [ ]:
# 13. Smoke test
def make_subset(root, names, tag):
    sub = DIRS["local_tmp"] / f"smoke_{tag}"
    shutil.rmtree(sub, ignore_errors=True)
    for split, n in (("train", CFG["smoke"]["train_images"]), ("val", CFG["smoke"]["val_images"]),
                     ("test", CFG["smoke"]["test_images"])):
        imgs = split_files(root, split)
        pick = random.Random(CFG["seed"]).sample(imgs, min(n, len(imgs)))
        (sub / "images" / split).mkdir(parents=True)
        (sub / "labels" / split).mkdir(parents=True)
        for p in pick:
            try:
                os.link(p, sub / "images" / split / p.name)
            except OSError:
                shutil.copy2(p, sub / "images" / split / p.name)
            shutil.copy2(Path(root) / "labels" / split / f"{p.stem}.txt", sub / "labels" / split / f"{p.stem}.txt")
    return sub


SMOKE_FP = stable_hash({"smoke": CFG["smoke"], "train": TRAIN_FP, "eval": CFG["eval"]})
smoke_marker = DIRS["smoke"] / f"smoke_passed_{SMOKE_FP}.json"
SMOKE = json.loads(smoke_marker.read_text()) if smoke_marker.exists() else {"per_experiment": {}}
todo = [e for e in EXPERIMENTS if e["status"] == "RUN" and e["id"] in PREP and e["id"] not in SMOKE["per_experiment"]]
if not todo:
    print("Smoke test already passed for this configuration and every available experiment:", smoke_marker)
else:
    print("Smoke testing:", [e["id"] for e in todo],
          "| already passed earlier:", sorted(SMOKE["per_experiment"]) or "none")
    for e in todo:
        print(f"\n=== SMOKE {e['id']} {e['variant']}")
        tag = f"{e['id']}_{e['variant']}"
        sub = make_subset(PREP[e["id"]]["root"], e["names"], tag)
        # tiny random subsets may lack rare classes: a warning, not a smoke failure
        rep, counts = pretrain_checks(sub, e["names"], f"SMOKE_{tag}", {"full_verify_images": True, "decode_sample": 10},
                                      require_train_classes=False)
        rep.finish()
        ty, ey = write_yamls(sub, e["names"], f"SMOKE_{tag}")
        EVAL_IMGSZ = CFG["smoke"]["imgsz"]
        t0 = time.time()
        rd = train_or_resume(e["id"], e["variant"], ty, DIRS["smoke"], smoke=True)
        train_s = time.time() - t0
        epochs_df, tsum = training_record(rd)
        best = rd / "weights" / "best.pt"
        val_res = evaluate_split(best, ey, e["names"], "val", rd, counts)
        test_res = evaluate_split(best, ey, e["names"], "test", rd, counts)
        speed = benchmark_speed(best, split_files(sub, "test"))
        stats = model_stats(best)
        (rd / "report").mkdir(exist_ok=True)
        plot_curves(epochs_df, rd / "report" / "curves.png", f"SMOKE {e['id']} (not a result)")
        save_prediction_grid(best, split_files(sub, "test"), rd / "report" / "test_predictions.png", rd)
        rec = build_report({**e}, rd, counts, tsum, epochs_df, val_res, test_res, speed, stats, smoke=True)
        chk = Report(f"SMOKE_{tag}_outputs", rd / "report")
        for f in ("weights/best.pt", "weights/last.pt", "results.csv", "epoch_log.jsonl", "run_status.json",
                  "report/metrics.json", "report/per_class_metrics.csv", "report/epochs.csv",
                  "report/training_config.json", "report/curves.png", "report/test_predictions.png"):
            chk.check(f"output exists: {f}", (rd / f).exists())
        chk.check("confusion matrix + PR curve saved (test)", (Path(test_res["save_dir"]) / "confusion_matrix.png").exists()
                  and (Path(test_res["save_dir"]) / "BoxPR_curve.png").exists())
        chk.check("epochs logged == epochs run", len(epochs_df) == CFG["smoke"]["epochs"], len(epochs_df))
        chk.check("metrics are finite numbers in [0, 1]",
                  all(0 <= test_res["overall"][k] <= 1 for k in ("precision", "recall", "f1", "mAP50", "mAP50_95")))
        chk.check("Ultralytics instance counts == our label counts", not test_res["instance_count_mismatch"]
                  and not val_res["instance_count_mismatch"], {**val_res["instance_count_mismatch"], **test_res["instance_count_mismatch"]})
        chk.check("speed benchmark produced timings", speed.get("images_timed", 0) > 0, speed.get("images_timed"))
        chk.finish()
        imgs_per_s = CFG["smoke"]["train_images"] * CFG["smoke"]["epochs"] / max(tsum["training_time_sum_epochs_s"] or train_s, 1e-6)
        scale = (CFG["train"]["imgsz"] / CFG["smoke"]["imgsz"]) ** 2
        est_h = PREP[e["id"]]["counts"]["train"]["images"] * CFG["train"]["epochs"] * scale / imgs_per_s / 3600
        SMOKE["per_experiment"][e["id"]] = {"run_dir": str(rd), "train_images_per_s_smoke": imgs_per_s,
                                            "rough_full_training_estimate_h": est_h}
        SMOKE.update(passed=True, utc=datetime.now(timezone.utc).isoformat(), env=ENV)
        write_json(smoke_marker, SMOKE)   # saved after each experiment, so a disconnect keeps finished ones
print("\nSMOKE TEST PASSED - pipeline trains, validates, tests, benchmarks and reports.")
for k, v in SMOKE["per_experiment"].items():
    print(f"  {k}: rough full-training estimate ~{v['rough_full_training_estimate_h']:.1f} h for "
          f"{CFG['train']['epochs']} epochs (very rough: tiny-subset throughput scaled by image area; "
          f"excludes validation time; early stopping may shorten it)")

## Experiments
Each experiment cell calls the same `run_experiment()`, which runs prepare → train or resume → evaluate → report. **Rerun a cell after a disconnect** and it resumes; an experiment that has finished is not retrained, and the cell only redisplays its stored report.

**Inspect after each experiment:**
* the loss curves: train and val should both fall, and a rising val loss signals overfitting;
* the **best epoch** and whether training stopped early;
* the effective batch, flagged if out-of-memory recovery changed it;
* test P / R / F1 / mAP and the per-class table;
* the confusion matrix and the prediction grid (in `plots/`).

In [ ]:
# Experiment runner (shared by E1-E4)
from IPython.display import display


def run_experiment(exp_id):
    e = next(x for x in EXPERIMENTS if x["id"] == exp_id)
    if e["status"] == "SKIPPED":
        print(f"{exp_id} SKIPPED - {e['reason']}\nNo labels are fabricated; this dataset cannot support a detection experiment.")
        rebuild_global_tables()
        return None
    if e["status"] == "DATA_UNAVAILABLE":
        print(f"{exp_id} NOT RUN - data unavailable: {e['reason']}")
        rebuild_global_tables()
        return None
    assert e["id"] in SMOKE.get("per_experiment", {}), f"Run the smoke test (cell 13) for {exp_id} first."
    assert e["id"] in PREP, "Run cell 11 first."
    global EVAL_IMGSZ
    EVAL_IMGSZ = CFG["train"]["imgsz"]
    p = PREP[e["id"]]
    rd = train_or_resume(e["id"], e["run_variant"], p["train_yaml"], DIRS["checkpoints"])
    epochs_df, tsum = training_record(rd)
    notes = []
    if tsum.get("stopped_by_user_at_epoch"):
        notes.append(f"DEVIATION: training stopped at epoch {tsum['stopped_by_user_at_epoch']} of "
                     f"{tsum['epochs_planned']} because the free Colab GPU quota ran out (finish_early). best.pt is "
                     "the best validation checkpoint among the completed epochs; this run did not get the full "
                     "epoch budget of the other experiments.")
    if not torch.cuda.is_available():
        notes.append("Speed was measured on CPU (no GPU available at evaluation time); it is not comparable with "
                     "experiments timed on a GPU.")
    if tsum["effective_batch"] != CFG["train"]["batch"]:
        notes.append(f"DEVIATION: effective batch {tsum['effective_batch']} != configured {CFG['train']['batch']} "
                     "(Ultralytics out-of-memory recovery).")
    if tsum["best_epoch_by_callback_log"] is not None and tsum["best_epoch_by_callback_log"] != tsum["best_epoch_by_results_csv"]:
        notes.append(f"best epoch: callback log says {tsum['best_epoch_by_callback_log']}, results.csv argmax says "
                     f"{tsum['best_epoch_by_results_csv']}")
    if p.get("subset"):
        notes.append(f"Trained on a fixed random subset of {p['subset']['n']} of {p['subset']['of']} training images "
                     f"(seed {p['subset']['seed']}, list: {p['subset']['list_file']}); validation and test are complete.")
    if e["variant"].startswith("bdd100k"):
        notes.append("Pretrained COCO head rows for car/bus/truck/motorcycle are remapped by class name by Ultralytics "
                     "8.4; UA-DETRAC's single 'vehicle' class has no COCO counterpart.")
    st = read_status(rd)
    mfile = rd / "report" / "metrics.json"
    if st["state"] == "EVALUATED" and mfile.exists():
        rec = json.loads(mfile.read_text())
        print(f"{exp_id}: already evaluated - showing stored report.")
    else:
        best = rd / "weights" / "best.pt"
        print(f"\n{exp_id}: evaluating best.pt (selected on validation) ...")
        val_res = evaluate_split(best, p["eval_yaml"], e["names"], "val", rd, p["counts"])
        test_res = evaluate_split(best, p["eval_yaml"], e["names"], "test", rd, p["counts"])   # held-out, once
        for res in (val_res, test_res):
            if res["instance_count_mismatch"]:
                notes.append(f"{res['overall']['split']}: Ultralytics instance counts differ from label files: "
                             f"{res['instance_count_mismatch']}")
        speed = benchmark_speed(best, split_files(p["root"], "test"))
        stats = model_stats(best)
        (rd / "report").mkdir(exist_ok=True)
        plot_curves(epochs_df, rd / "report" / "curves.png", f"{exp_id} {e['variant']} - YOLOv8n")
        save_prediction_grid(best, split_files(p["root"], "test"), rd / "report" / "test_predictions.png", rd)
        rec = build_report({**e, "notes": notes, "subset": p.get("subset")}, rd, p["counts"], tsum, epochs_df,
                           val_res, test_res, speed, stats)
    tr = rec["training"]
    print(f"\n{exp_id} {e['variant']}  | best epoch {tr['best_epoch']} of {tr['epochs_completed']} "
          f"({'early-stopped' if tr['early_stopped'] else 'full budget'}) | batch {tr['effective_batch']} | "
          f"training {(tr['training_time_sum_epochs_s'] or 0) / 3600:.2f} h over {tr['sessions']} session(s)")
    print("best checkpoint:", tr["best_checkpoint"])
    for n in rec["notes"]:
        print("NOTE:", n)
    cols = ["split", "class", "images", "instances", "precision", "recall", "f1", "mAP50", "mAP50_95"]
    display(pd.DataFrame([rec["validation"]["overall"], rec["test"]["overall"]])[cols])
    display(pd.DataFrame(rec["test"]["per_class"])[cols])
    print("speed:", {k: round(v, 3) if isinstance(v, float) else v for k, v in rec["speed"].items()})
    print("model:", {k: round(v, 3) if isinstance(v, float) else v for k, v in rec["model_stats"].items()})
    print("report folder:", rd / "report")
    display(rebuild_global_tables())
    return rec

### E1 · UA-DETRAC mirror (1 class)

In [ ]:
# E1
RESULT_E1 = run_experiment("E1")

### E2 · BDD100K (4 classes)

In [ ]:
# E2
RESULT_E2 = run_experiment("E2")

### E3 · QMUL Junction

In [ ]:
# E3
RESULT_E3 = run_experiment("E3")

### E4 · AI City Challenge 2021

In [ ]:
# E4
RESULT_E4 = run_experiment("E4")

## R · Final report 🔍
This rebuilds and shows the cross-experiment summary from the files on Drive, not from memory. An experiment is listed with metrics **only** if its status is `EVALUATED`; failed, unfinished and skipped ones show their status and reason.

Remember when reading the results:
* **E1 and E2 differ in classes, domain and size.** Their numbers describe each dataset; they are not a ranking of datasets.
* **Speed is measured on the Colab GPU at full precision, batch 1.** It is not the paper's Jetson / TensorRT setting.
* **YOLOv8n alone doesn't reproduce the paper's model comparison.**

In [ ]:
# R. Final report
summary = rebuild_global_tables()
display(summary)
print("Machine-readable files:")
for f in ("summary_all_experiments.csv", "summary_all_experiments.json", "per_class_all_experiments.csv"):
    p = DIRS["metrics"] / f
    print(f"  {'OK ' if p.exists() else '-- '}{p}")
not_done = summary[(summary["status"] != "EVALUATED") & (summary["status"] != "SKIPPED")]
if len(not_done):
    print("\nNOT FINISHED (no metrics reported):", not_done[["experiment", "status", "reason"]].to_dict("records"))